# Reinforcement Learning

# 7. Parametric Bandits

The objective of this lab is to recommend contents (here movies) using **parametric bandits**. The rewards are binary (like or dislike).

Please read the instructions:
* Write concise code and text.
* Check that your notebook runs without errors.
* Clear all cell outputs before upload on Moodle.

## Imports

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

You will need ``ipywidgets`` to simulate the interactions with the user.

In [ ]:
#!pip install ipywidgets

In [ ]:
from ipywidgets import AppLayout, Button, GridspecLayout, Image, Layout

In [ ]:
#!pip install scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer

## Data

We work on a catalogue of 1037 movies available in 2015.

In [ ]:
catalogue = pd.read_pickle('movie_database.pickle')

In [ ]:
len(catalogue)

In [ ]:
catalogue.head()

The features are the following:

|Column|Description|Type|
|:---|:---|:---|
|Actors| Actors staring | list of strings|
|Awards| Awards received| string|
|Country| Country of origin| list of strings|
|Director| Director(s) of the movie|  list of strings|
|Genre| Genres (Action, ...) | list of strings|
|Language| Language(s) spoken |list of strings|
|Rated| Public rating (G = General, R = Restricted, ...)| list of strings|
|Released| Date of the movie| date|
|Title|Title of the movie|string|
|imdbID| IMDB id| string|
|imdbRating| IMDB rating (between 0 and 10)| float|
|Metascore| Metacritic score (between 0 and 100)|float|
|Box_office| Total money generated|float|
|imdbVotes| Number of IMDB votes| float|
|Runtime| Duration of the movie (in minutes)|float|
|poster| Poster of the movie (jpg)| binary string|

In [ ]:
# Display the posters

def get_poster(k, scale=1):
    """Get poster of a movie."""
    return Image(
        value = catalogue.loc[k].poster,
        format = 'jpg',
        width = 130 * scale,
        height = 200 * scale,
    )

def display_posters(index=None, n_col=5, n_rows=4):
    """Display posters on the specified number of columns and rows in order given by the index."""
    if index is None:
        index = np.arange(len(catalogue))
    if len(index):
        n_rows = min(n_rows, int(np.ceil(len(index) / n_col)))
        grid = GridspecLayout(n_rows, n_col)
        k = 0
        for i in range(n_rows):
            for j in range(n_col):
                if k < len(index):
                    grid[i, j] = get_poster(index[k])
                k += 1 
        return grid

In [ ]:
display_posters()

## Features

We will describe each movie by some features, for instance its genre.

In [ ]:
mlb = MultiLabelBinarizer()

In [ ]:
movies = pd.DataFrame(mlb.fit_transform(catalogue['Genre']), columns=mlb.classes_)

In [ ]:
movies.head()

In [ ]:
movies.columns

## User

Each user will be modeled by a vector of weights (positive or negative) on each feature. 

In [ ]:
user = pd.DataFrame(0, index = [0], columns=movies.columns)
user['Action'] = 2
user['Crime'] = 1
user['Sci-Fi'] = -2

## To do

* Display the favorite movies of this user. <br>**Hint:** Use a dot product.
* Test another user, and quantify their similarity (e.g., [Jaccard index](https://en.wikipedia.org/wiki/Jaccard_index) of top-100 movies).

In [ ]:
theta = user.values.reshape(-1)  
X = movies.values.astype(float)         

scores = X @ theta   
top_k = 20 

top_idx = np.argsort(scores)[::-1][:top_k]

catalogue.loc[top_idx, ["Title", "Genre"]].assign(score=scores[top_idx])

In [ ]:
user2 = pd.DataFrame(0, index=[0], columns=movies.columns)
user2["Comedy"] = 2
user2["Romance"] = 1
user2["Horror"] = -2

theta2 = user2.values.reshape(-1)

scores2 = X @ theta2

K = 100
top1 = set(np.argsort(scores)[::-1][:K])
top2 = set(np.argsort(scores2)[::-1][:K])

jaccard = len(top1 & top2) / len(top1 | top2)
jaccard

### Remark:

The recommendations are coherent with the user’s preferences. The low Jaccard index between two users with different genre weights shows that the model correctly captures dissimilar tastes.

## Offline learning

We start with offline learning. There are 2 steps: 
1. Collect the user's opinion on a few movies (e.g., 10)
2. Rank the other movies by logistic regression.

Let's test that.

In [ ]:
# Add a column to record the user's feedback (like / dislike)
movies = movies.assign(feedback=None)

In [ ]:
# Select a random movie (not yet seen by the user)
    
def select_random_movie():
    index = np.flatnonzero(movies.feedback.isna())
    if len(index):
        return np.random.choice(index)
    else:
        return np.random.choice(len(movies))

In [ ]:
# Create buttons

def create_expanded_button(description, button_style):
    return Button(
        description=description,
        button_style=button_style,
        layout=Layout())

def update_feedback(button):
    global movie_id
    movies.loc[movie_id, 'feedback'] = button.description == 'like'
    
def update_poster():
    global movie_id
    img.value = catalogue.loc[movie_id].poster
    
def on_button_clicked(button):
    global movie_id
    update_feedback(button)
    movie_id = select_random_movie()
    update_poster()    

In [ ]:
# Setting the buttons
left_button = create_expanded_button('like', 'success')
right_button = create_expanded_button('dislike', 'danger')
left_button.on_click(on_button_clicked)
right_button.on_click(on_button_clicked)

# Setting the movie poster
movie_id = select_random_movie()
img = get_poster(movie_id, scale=1.5)

# Display
AppLayout(
    left_sidebar=left_button,
    right_sidebar=right_button, 
    center=img,
    pane_widths=[0.3, 0.4, 0.3]
)

## To do

* Give your opinion on 5-10 movies, making sure that you get at least one like and at least one dislike.
* Apply logistic regression and display the other movies in order of preference (top movies first).
* Give your top-3 and bottom-3 genres, as predicted by the model.

In [ ]:
# likes
likes = np.flatnonzero(movies.feedback==True)
display_posters(likes)

In [ ]:
# dislikes
dislikes = np.flatnonzero(movies.feedback==False)
display_posters(dislikes)

In [ ]:
n_like = (movies.feedback == True).sum()
n_dislike = (movies.feedback == False).sum()
n_total = movies.feedback.notna().sum()
n_like, n_dislike, n_total

In [ ]:
model = LogisticRegression(fit_intercept=False)

In [ ]:
train_idx = movies.feedback.notna()

X_train = movies.loc[train_idx, movies.columns.drop("feedback")].values
y_train = movies.loc[train_idx, "feedback"].astype(int).values

model = LogisticRegression(fit_intercept=False, solver="lbfgs", max_iter=200)
model.fit(X_train, y_train)

In [ ]:
X_all = movies.drop(columns=["feedback"]).values

p_like = model.predict_proba(X_all)[:, 1]

unseen_idx = np.flatnonzero(movies.feedback.isna())

ranked_unseen = unseen_idx[np.argsort(p_like[unseen_idx])[::-1]]

top_k = 20
top_idx = ranked_unseen[:top_k]

catalogue.loc[top_idx, ["Title", "Genre"]].assign(score=p_like[top_idx])

display_posters(index=top_idx, n_col=5, n_rows=int(np.ceil(top_k/5)))

In [ ]:
catalogue.loc[top_idx, ["Title", "Genre"]].assign(score=p_like[top_idx]).head(20)

In [ ]:
w = model.coef_.reshape(-1)

genre_names = movies.columns.drop("feedback")

coef = pd.Series(w, index=genre_names).sort_values(ascending=False)

top3 = coef.head(3)
bottom3 = coef.tail(3)

top3, bottom3

### Remark:

The offline logistic regression successfully ranks unseen movies by predicted preference. The learned coefficients indicate the most liked genres (Romance, Action, Crime) and the least liked ones (Western, Fantasy, Sci-Fi).

## Online learning

We now learn the user preferences online, as they come. For that, we use a Bayesian algorithm inspired by Thompson sampling. 

On each feedback provided by the user:
1. (Learn) The parameter (vector of weights) is learned.
2. (Sample) A new parameter is sampled, assuming a Gaussian distribution.
3. (Act) The top movie for this new parameter, among movies not yet seen by the user, is proposed. 

Note that:
* In step 1, we retrain the estimator **from scratch**, using logistic regression on all training data samples (**no** online estimation).
* In step 2, we discard correlations (**diagonal** covariance matrix).

## To do

* Complete the function ``select_bayes`` below.
* Test it on 5-10 movies, until you get a few likes and a few dislikes.
* Display the other movies in order of preference (top movies first).

In [ ]:
def select_bayes():    
    """Return the top movie id according to the sampled parameter."""
    
    if set(movies.feedback) == {True, False, None}:
        
        # to be completed (learn, sample, act)
        # learn
        train_idx = movies.feedback.notna()
        X_train = movies.loc[train_idx, movies.columns.drop("feedback")].values
        y_train = movies.loc[train_idx, "feedback"].astype(int).values

        model = LogisticRegression(fit_intercept=False, solver="lbfgs", max_iter=200)
        model.fit(X_train, y_train)

        theta_hat = model.coef_.reshape(-1)

        # sample
        p = model.predict_proba(X_train)[:, 1]
        w = p * (1 - p)  # weights

        fisher_diag = (w[:, None] * (X_train ** 2)).sum(axis=0)
        lam = 1.0
        var = 1.0 / (lam + fisher_diag)

        theta_sample = np.random.normal(theta_hat, np.sqrt(var))

        # act
        X_all = movies.drop(columns=["feedback"]).values
        unseen_idx = np.flatnonzero(movies.feedback.isna())

        scores = X_all[unseen_idx] @ theta_sample
        return int(unseen_idx[np.argmax(scores)])
    
    else:    
        return select_random_movie()    

In [ ]:
# reset
movies = movies.assign(feedback=None)

In [ ]:
def on_button_clicked(button):
    global movie_id
    update_feedback(button)
    movie_id = select_bayes()
    update_poster()    

In [ ]:
# Setting the buttons
left_button = create_expanded_button('like', 'success')
right_button = create_expanded_button('dislike', 'danger')
left_button.on_click(on_button_clicked)
right_button.on_click(on_button_clicked)

# Setting the movie poster
movie_id = select_random_movie()
img = get_poster(movie_id, scale=1.5)

# Display
AppLayout(
    left_sidebar=left_button,
    right_sidebar=right_button, 
    center=img,
    pane_widths=[0.3, 0.4, 0.3]
)

In [ ]:
(movies.feedback == True).sum(), (movies.feedback == False).sum(), movies.feedback.notna().sum()

In [ ]:
n_like = (movies.feedback == True).sum()
n_dislike = (movies.feedback == False).sum()
print("likes/dislikes =", n_like, n_dislike)

if (n_like == 0) or (n_dislike == 0):
    print("Need at least one like and one dislike to rank movies.")
else:
    train_idx = movies.feedback.notna()
    X_train = movies.loc[train_idx, movies.columns.drop("feedback")].values
    y_train = movies.loc[train_idx, "feedback"].astype(int).values

    model = LogisticRegression(fit_intercept=False, solver="lbfgs", max_iter=200)
    model.fit(X_train, y_train)

    X_all = movies.drop(columns=["feedback"]).values
    p_like = model.predict_proba(X_all)[:, 1]

    unseen_idx = np.flatnonzero(movies.feedback.isna())
    ranked_unseen = unseen_idx[np.argsort(p_like[unseen_idx])[::-1]]

    top_k = 20
    top_idx = ranked_unseen[:top_k]

    display(catalogue.loc[top_idx, ["Title", "Genre"]].assign(score=p_like[top_idx]))
    display(display_posters(index=top_idx, n_col=5, n_rows=int(np.ceil(top_k/5))))


### Remark:

After a few online interactions (6 likes, 3 dislikes), the model learns the user’s preferences and ranks unseen movies by predicted like probability.

## Analysis

Finally, we would like to assess the quality of our bandit algorithm.

## To do

* Choose a user, that is a parameter $\theta$ (vector of weights).
* Simulate the feedback of this user (like / dislike) for 100 movies proposed by the algorithm, assuming binary rewards with mean
$$
q(a) = \frac 1 {1 + e^{-\theta^T a}}
$$
where $a$ is the action (that is, the movie).
* Compute the [Spearman's correlation coefficient](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient) of the ranking of the unseen movies provided by the algorithm, compared to the ground-truth ranking.
* Plot the evolution of this coefficient with respect to the number of movies seen by the user, from 1 to 100.
* Give the top-3 and bottom-3 genres, as predicted by the model, and compare to the ground-truth.
* (Optional) Do the same experiments with other features (e.g., actors, actors + genres, actors + director + genres).

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def spearman_corr(x, y):
    # Spearman = corrélation de Pearson entre les rangs
    rx = pd.Series(x).rank(method="average").values
    ry = pd.Series(y).rank(method="average").values
    return np.corrcoef(rx, ry)[0, 1]

In [ ]:
theta_true = user.values.reshape(-1)  # ground-truth user
X_all = movies.drop(columns=["feedback"], errors="ignore").values.astype(float)  # features des films
feature_names = movies.drop(columns=["feedback"], errors="ignore").columns

In [ ]:
T = 100
np.random.seed(0)

movies = movies.assign(feedback=None)

spearman_vals = []

for t in range(1, T + 1):

    a = select_bayes()

    p = sigmoid(theta_true @ X_all[a])
    r = (np.random.rand() < p)
    movies.loc[a, "feedback"] = r

    unseen_idx = np.flatnonzero(movies.feedback.isna())

    n_like = (movies.feedback == True).sum()
    n_dislike = (movies.feedback == False).sum()

    if (n_like == 0) or (n_dislike == 0) or (len(unseen_idx) < 5):
        spearman_vals.append(np.nan)
        continue

    train_idx = movies.feedback.notna()
    X_train = movies.loc[train_idx, movies.columns.drop("feedback")].values
    y_train = movies.loc[train_idx, "feedback"].astype(int).values

    model_tmp = LogisticRegression(fit_intercept=False, solver="lbfgs", max_iter=200)
    model_tmp.fit(X_train, y_train)
    theta_hat = model_tmp.coef_.reshape(-1)

    score_alg = X_all[unseen_idx] @ theta_hat
    score_true = X_all[unseen_idx] @ theta_true

    spearman_vals.append(spearman_corr(score_alg, score_true))

In [ ]:
plt.figure()
plt.plot(np.arange(1, T + 1), spearman_vals)
plt.xlabel("Number of movies seen")
plt.ylabel("Spearman correlation (unseen ranking)")
plt.ylim(-1, 1)
plt.show()

In [ ]:
n_like = (movies.feedback == True).sum()
n_dislike = (movies.feedback == False).sum()

if (n_like == 0) or (n_dislike == 0):
    print("Not enough likes/dislikes to fit the final model.")
else:
    train_idx = movies.feedback.notna()
    X_train = movies.loc[train_idx, movies.columns.drop("feedback")].values
    y_train = movies.loc[train_idx, "feedback"].astype(int).values

    model_final = LogisticRegression(fit_intercept=False, solver="lbfgs", max_iter=200)
    model_final.fit(X_train, y_train)

    w_pred = pd.Series(model_final.coef_.reshape(-1), index=feature_names).sort_values(ascending=False)
    w_true = pd.Series(theta_true, index=feature_names).sort_values(ascending=False)

    print("Predicted top-3:\n", w_pred.head(3))
    print("Predicted bottom-3:\n", w_pred.tail(3))
    print("Ground-truth top-3:\n", w_true.head(3))
    print("Ground-truth bottom-3:\n", w_true.tail(3))

### Remark:

The learned coefficients are consistent with the ground-truth: Action is identified as the most preferred genre and Sci-Fi as the least preferred one. The remaining differences are expected since most ground-truth weights are zero and the model is fitted from noisy binary feedback on a limited set of interactions.